# 🌳 NLP Playground — 重量級3モデル比較（無料Colab版）
スマホのブラウザからGoogle Colabを開き、**spaCy + RoBERTa / Stanza / BERT POS** の本物のPythonモデルを実行するノートです。

- **料金の発生するAPI・Cloud Run・課金登録は使用しません。** Colabの無料枠を使います。自分で有料プランに登録しないでください。
- 「ランタイム」→「すべてのセルを実行」でも、各セルの ▶ を上から順番に押してもOK。
- **ランタイムの種類は「CPU」を推奨**。無料枠のランタイムは予告なく終了することがあります。
- 3つのモデルは**それぞれ異なるタグ体系・トークン化規則**を使います。品詞の正誤やモデル性能を単純には比較できません。
- 初回はパッケージとモデルをダウンロードします。処理が重いので、モデルは一つずつ読み込んでメモリを解放します。

公式: [spaCy](https://spacy.io/models/en#en_core_web_trf) / [Stanza](https://stanfordnlp.github.io/stanza/constituency.html) / [BERT POS](https://huggingface.co/Alireza0017/bert-finetuned-pos)

## ① 準備：ライブラリとspaCyのTransformerモデルを導入
初回起動やColabの仮想環境が再作成されたときは再実行が必要です。

In [ ]:
%%capture
%pip install -q "spacy>=3.8,<3.9" "spacy-transformers>=1.3,<1.4" "stanza>=1.10" "transformers>=4.50,<5" pandas
!python -m spacy download en_core_web_trf

## ② 実験する英文を入力
フォームの文を書き換えて、このセルから再実行してね。

In [ ]:
sentence = "The old man the boats." #@param {type:"string"}
assert sentence.strip(), "英文を入力してね"
print("実験文：", sentence)

## ③ spaCy + RoBERTa（en_core_web_trf）
**品詞**（UPOSと詳細タグ）・**依存構造**を表示。句構造ツリーは出力しません。

In [ ]:
import gc
import pandas as pd
import spacy
spacy.require_cpu()
spacy_nlp = spacy.load("en_core_web_trf")
spacy_doc = spacy_nlp(sentence)
spacy_rows = [{"token": t.text, "UPOS": t.pos_, "細分類": t.tag_, "lemma": t.lemma_, "dependency": t.dep_, "head": t.head.text} for t in spacy_doc]
display(pd.DataFrame(spacy_rows))
del spacy_doc, spacy_nlp
gc.collect()

## ④ Stanza（品詞・依存構造・句構造ツリー）
標準モデルとTransformerを使う高精度モデル（`default_accurate`）を切り替え可能にしました。**まずは標準版**で動作を確かめ、高精度版を試すときはチェックしてこのセルを再実行してね。高精度版はメモリ・モデルダウンロードがさらに重くなります。

In [ ]:
use_accurate_stanza = False #@param {type:"boolean"}
stanza_package = "default_accurate" if use_accurate_stanza else "default"
import stanza
print("Stanza package:", stanza_package)
stanza.download("en", processors="tokenize,pos,lemma,depparse,constituency", package=stanza_package, verbose=False)
stanza_nlp = stanza.Pipeline("en", processors="tokenize,pos,lemma,depparse,constituency", package=stanza_package, use_gpu=False, verbose=False)
stanza_doc = stanza_nlp(sentence)
stanza_rows = []
stanza_trees = []
for sent in stanza_doc.sentences:
    for w in sent.words:
        stanza_rows.append({"token": w.text, "UPOS": w.upos, "細分類": w.xpos, "lemma": w.lemma, "head_id": w.head, "dependency": w.deprel})
    stanza_trees.append(str(sent.constituency))
display(pd.DataFrame(stanza_rows))
print("【Stanza 句構造ツリー】")
for tree in stanza_trees: print(tree, "\n")
del stanza_doc, stanza_nlp
gc.collect()

## ⑤ BERT（CoNLL-2003で品詞タグ付け用に調整したモデル）
このBERTはspaCyやStanzaと異なり、**品詞タグ付け専用**で、依存構造や句構造は出しません。サブワードを結合した結果を表示します。

In [ ]:
from transformers import pipeline
bert_tagger = pipeline("token-classification", model="Alireza0017/bert-finetuned-pos", aggregation_strategy="first", device=-1)
bert_predictions = bert_tagger(sentence)
bert_rows = [{"token": p["word"], "POSラベル": p["entity_group"], "score": round(float(p["score"]),4), "start": p.get("start"), "end": p.get("end")} for p in bert_predictions]
display(pd.DataFrame(bert_rows))
del bert_predictions, bert_tagger
gc.collect()

## ⑥ 生のデータをJSONに保存
異なるモデルの結果を**同じ品詞体系であるかのように扱わず**、それぞれの原出力に近い形で保存します。ボタンを押してダウンロードできます。

In [ ]:
import json
result = {"sentence":sentence, "models":{"spacy_en_core_web_trf":spacy_rows, "stanza_"+stanza_package: {"tokens":stanza_rows, "constituency":stanza_trees}, "bert_finetuned_pos":bert_rows}}
with open("nlp-comparison.json","w",encoding="utf-8") as fp:
    json.dump(result,fp,ensure_ascii=False,indent=2)
print("✅ nlp-comparison.json を作成したよ。ダウンロードしたい場合は次のセルを実行してね。")

In [ ]:
from google.colab import files
files.download("nlp-comparison.json")

## 注意・制約
- Colabは**無料でも利用可能**ですが、コンピューティング資源は保証されず、制限・セッション切断があります。Notebookのセル内で実験する用途として利用してください。
- スマホの画面が小さければ、横向きやChromeの「PC版サイト」で表を見てね。
- spaCyのRoBERTa、Stanzaの標準モデル、BERT POSは**学習データも出力体系も異なる**ため、単純に正解率を比較できません。
- BERTはモデルの確信度が高くても正しいとは限りません。
- このノートは試作版です。環境側のパッケージ更新で動かなくなる場合は報告してください。